In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%python
dbutils.widgets.text("catalogo", "catalog_gamd")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
%python
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

### Transformación

#### Generar catalogo de eval set

In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


data =  [(1, 'Prior'),
(2, 'Train')]

schema = StructType([
StructField("id_eval_set", IntegerType(), True),
StructField("eval_set", StringType(), True)
])

df_eval_set = spark.createDataFrame(data=data, schema=schema)

df_eval_set.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.cat_eval_set")

#### Tabla order_products

Aplicamos union de tabal prior y train, agregando en cada una columna id_eval_set para obtener el dato de cat_eval_set

In [0]:
%python
# leer tablas order products
df_prior = spark.table("catalog_gamd.bronze.order_products_prior")
df_train = spark.table("catalog_gamd.bronze.order_products_train")

In [0]:
%python

new_columns = ["id_eval_set"]
id_eval_value = 1

for col in new_columns:
  df_prior = df_prior.withColumn(col, lit(id_eval_value))

#df_prior.withColumn("id_eval_set", lit(1))
df_prior.display()

In [0]:
%python

new_columns = ["id_eval_set"]
id_eval_value = 2

for col in new_columns:
  df_train = df_train.withColumn(col, lit(id_eval_value))

#df_prior.withColumn("id_eval_set", lit(1))
df_train.display()

In [0]:
%python
df_orders = df_prior.union(df_train)

df_orders.write.format("delta").mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.order_products")

#### Tabla orders

In [0]:
%python
# leer tablas orders
df_orders_final = spark.table(f"{catalogo}.{esquema_source}.orders")

In [0]:
%python
from pyspark.sql.functions import when, lit

df_orders_final = df_orders_final.withColumn("id_eval_set", when(df_orders_final.eval_set == "prior", lit(1)).otherwise(lit(2)),                     
                     ).select("order_id", "user_id",  "id_eval_set", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order")

# sanear nulos en days_since_prior_order
avg_value = df_orders_final.select("days_since_prior_order").groupBy().avg().collect()[0][0]
df_orders_final = df_orders_final.withColumn("days_since_prior_order", when(df_orders_final.days_since_prior_order.isNull(), avg_value).otherwise(df_orders_final.days_since_prior_order))

In [0]:
%python
# ingestar datos en orders
df_orders_final.write.format("delta").mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.orders")